# Stability and the CFL condition

In the [previous lesson](./06-1d-convection.ipynb), we studied the numerical solution of the linear and non-linear convection equations, using the finite-difference method. 
We began by discretizing the one-way wave equation using a classic *forward-time/backward space* scheme, and computing the solution using an initial condition consisting of a square pulse.

Computing with finer spatial grids while keeping the time step constant, you encountered four cases:

1. a coarse run with `nx=41` smoothed the square pulse and attenuated its height,
2. a finer run with `nx=81` improved the solution, but smoothing still ocurred,
3. a surprising case with `nx=101`matched the exact solution, and
4. a further refinement with `nx=121` destroyed the solution.

In this lesson, we will explore why changing the discretization parameters can affect your solution in such a drastic way.
The central question we want to answer is:

> Why did refining the spatial grid improve the linear-convection calculation, make it an exact grid shift at one setting, and then destroy it?

Let's begin by importing our favorite Python libraries for numerical computing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

The code below corresponds to the same function we used in the previous lesson, but incorporating the vectorized update in space, leaving only the loop in time. Be sure to review the way array slicing works in this code, sketched in [Figure %s](./06-1d-convection.ipynb#fig-vectorized-backward-difference) of [Lesson 6](./06-1d-convection.ipynb), and the Python refresher that came after.

In [ ]:
def advance_linear_convection(u0, c, dx, dt, num_steps):
    '''Advance linear convection with the FTBS scheme.'''
    u = u0.copy()
    for n in range(num_steps):
        u[1:] = u[1:] - c * dt / dx * (u[1:] - u[:-1])
    return u

## How far does the wave travel in one step?

Look again at the discretized equation implemented by our function:

$$
\label{eq-ftbs-courant-update}
u_i^{n+1}=u_i^n-\frac{c\Delta t}{\Delta x}
\left(u_i^n-u_{i-1}^n\right).
$$

Pay attention to the coefficient multiplying the spatial difference: `c * dt / dx`. The speed $c$ has units of distance per time, so $c\Delta t$ is the distance the exact wave travels during one time step. Dividing by the grid spacing expresses that distance in grid intervals. We give this dimensionless ratio a name:

$$
\label{eq-courant-number}
C=\frac{c\Delta t}{\Delta x}.
$$

This is the **Courant number**, also known as the **CFL number** (Courant–Friedrichs–Lewy) [@courant1928; English translation, -@courant1967]. For the positive speed considered here, $C=0.8$ means that the wave travels eight-tenths of a grid interval per time step; $C=1$ means one complete interval.

The ratio is already present in [Equation %s](#eq-ftbs-courant-update) and in our code. Refining the spatial grid while keeping the time step fixed makes $\Delta x$ smaller and the Courant number larger. What values did it take in the four runs from the previous lesson?

:::{warning .simple .dropdown icon=false open=false} On paper
Use $L=2$, $c=1$, and $\Delta t=0.02$, as in Lesson 6. For each of `nx = 41, 81, 101, 121`, calculate $\Delta x=L/(nx-1)$ and then $C$. Check that the units cancel. Interpret each result as the distance traveled in one step, measured in grid intervals, and match it to the behavior you observed.
:::

In [ ]:
L = 2.0
c = 1.0
dt = 0.02
nx_trials = [41, 81, 101, 121]

print(f"{'nx':>5} {'dx':>10} {'C':>8}")
for nx in nx_trials:
    dx = L / (nx - 1)
    courant = c * dt / dx
    print(f'{nx:5d} {dx:10.5f} {courant:8.3f}')

The four Courant numbers are $0.4$, $0.8$, $1.0$, and $1.2$. The run that shifted the pulse exactly had $C=1$; the failed run had $C=1.2$. These observations suggest that this ratio matters, but they do not yet explain the failure or establish a stability condition.

To investigate, we will ask a more focused question: **what happens to a small disturbance in the initial data?**

## Perturbation study

:::{warning .simple .dropdown icon=false open=false} In your notebook

Reconstruct this experiment in your own notebook, using the vectorized `advance_linear_convection()` function above. Before running it, predict how a disturbance of $10^{-3}$ will behave on each of the three grids. Build the paired initial states and record their maximum difference after every step. You may copy the plot formatting and table layout. Compare your results with your predictions, and explain what the experiment does and does not establish about stability.
:::

Imagine running the same calculation twice, with just one initial value changed by $10^{-3}$ in the second run. Both runs use the same grid, time step, method, and inflow value. Does their difference remain small as we advance in time, or does the calculation amplify it?

Use a constant background, $u=1$, so that the disturbance is easy to isolate. In the second array, add $10^{-3}$ at $x=0.5$. The unperturbed exact solution stays constant; the equation transports a disturbance without increasing its height. Our numerical update may behave differently.

We will repeat this paired experiment on `nx = 81, 101, 121`, keeping $c=1$ and $\Delta t=0.02$. These are the three runs with $C=0.8$, $1.0$, and $1.2$. Advance each pair for 25 steps, reaching the same physical time $T=0.5$ on every grid.

At each step, measure the largest absolute difference between the two arrays:

$$
\label{eq-perturbation-maximum-difference}
E^n=\max_i\left|u_{b,i}^n-u_{a,i}^n\right|.
$$

Here, $a$ denotes the unperturbed run and $b$ the perturbed run. Initially, $E^0=10^{-3}$. Taking absolute values matters because a disturbance can develop both positive and negative differences.

The perturbation occupies one grid point, so its physical width changes with the grid. We are testing how the method amplifies a grid disturbance; this is not a convergence comparison for one fixed continuous initial profile.

In [ ]:
num_steps = 25
perturbation = 1e-3
nx_perturbation = [81, 101, 121]
time = dt * np.arange(num_steps + 1)
error_histories = []
courant_numbers = []

Each run starts from fresh arrays. The point at `nx // 4` lies at $x=0.5$ on all three grids; `//` is integer division. The left boundary remains $u=1$ in both runs, because the function updates only `u[1:]`.

We call `advance_linear_convection()` with `num_steps=1` to inspect the difference after every step. This uses the same update as advancing all 25 steps at once, while letting us record its behavior along the way. The FTBS stencil can pass a disturbance at most one grid point to the right per step. Even on the coarsest grid, the perturbed point is more than 25 grid intervals from the outflow, so the disturbance cannot leave the array during this experiment.

**Before running:** predict which of the three cases will reduce, preserve, or amplify the maximum difference. Treat the prediction as a hypothesis to test.

In [ ]:
for nx in nx_perturbation:
    dx = L / (nx - 1)
    courant = c * dt / dx

    u_a = np.ones(nx)
    u_b = u_a.copy()
    u_b[nx // 4] += perturbation

    error = np.empty(num_steps + 1)
    error[0] = np.max(np.abs(u_b - u_a))
    for n in range(num_steps):
        u_a = advance_linear_convection(u_a, c, dx, dt, 1)
        u_b = advance_linear_convection(u_b, c, dx, dt, 1)
        error[n + 1] = np.max(np.abs(u_b - u_a))

    error_histories.append(error)
    courant_numbers.append(courant)

Let's plot the recorded differences. A logarithmic vertical scale lets us see shrinking and growing disturbances on the same axes. Equal vertical distances represent equal multiplicative changes, rather than equal additive changes.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.5))
for courant, error in zip(courant_numbers, error_histories):
    ax.semilogy(time, error, label=f'C = {courant:.2f}')

ax.axhline(perturbation, color='black', linestyle=':',
           label='Initial disturbance')
ax.set_xlabel('Time')
ax.set_ylabel('Maximum difference between runs')
ax.grid(which='both', alpha=0.3)
ax.legend()
fig.tight_layout();

A few selected steps make the amount of amplification easier to compare:

In [ ]:
steps_to_report = [1, 5, 10, 25]
print(f"{'C':>6}" + ''.join(
    f'{str(step) + " steps":>14}' for step in steps_to_report
))
for courant, error in zip(courant_numbers, error_histories):
    print(f'{courant:6.2f}' + ''.join(
        f'{error[step]:14.2e}' for step in steps_to_report
    ))

For $C=0.8$, the maximum difference decreases as the disturbance spreads over neighboring points. For $C=1$, its maximum stays at $10^{-3}$. For $C=1.2$, a difference initially one-thousandth of the background grows to about $1$ after only 25 steps. The calculation has turned a small change in the initial data into a large change in the result.

This is the question behind **numerical stability**: does the method control the amplification of small disturbances? More precisely, over a fixed physical time interval, we seek a bound on amplification that remains independent of the mesh as we refine it under the stated conditions. Stability need not mean that every disturbance decays; the $C=1$ result illustrates controlled propagation without decay.

These three runs provide evidence for particular choices of grid and time step. They do not prove a general bound. We still need to explain why the coefficient in [Equation %s](#eq-ftbs-courant-update) separates these behaviors.

:::{warning .simple .dropdown icon=false open=false} Self-checks
- For each Courant number, divide the final maximum difference by the initial maximum difference. What amplification factor do you obtain over $T=0.5$?
- Why would running only the constant, unperturbed state fail to expose the growing disturbance?
- Does the decreasing difference at $C=0.8$ establish that this grid gives an accurate transported pulse? Relate your answer to the smoothing observed in Lesson 6.
:::

## Where does the information come from?

We introduced the Courant number by asking how far the exact wave travels during one time step. Now turn that question around: **where was the information needed at $x_i$ one time step earlier?**

For constant positive speed $c$, the solution travels along a characteristic without changing its value. Tracing that characteristic backward from $(x_i,t_{n+1})$ gives

$$
\label{eq-cfl-characteristic-foot}
u(x_i,t_{n+1})=u(x_i-c\Delta t,t_n).
$$

The exact solution therefore needs the value at $x_i-c\Delta t$. Our numerical update, [Equation %s](#eq-ftbs-courant-update), uses only two old values: $u_{i-1}^n$ and $u_i^n$. [Figure %s](#fig-cfl-domain-of-dependence) puts these two descriptions on the same space–time diagram.

```{figure} ./figures/CFLcondition.png
:label: fig-cfl-domain-of-dependence
:alt: Space–time diagram with an FTBS update at i, n+1, its two input points at i-1 and i at time n, and a backward characteristic landing between those inputs
:width: 450px
:align: center

The FTBS update uses the two black points at time level $n$. The dashed characteristic traces the exact solution back a distance $c\Delta t$. The shaded triangle connects the numerical inputs to the new point; the illustrated characteristic lands between them.
```

In the case drawn, $0<C<1$: the characteristic lands between $x_{i-1}$ and $x_i$. At $C=1$, it lands exactly on $x_{i-1}$. For $C>1$, it lands farther to the left, outside the interval spanned by the two input points.

This is the local picture of a **domain of dependence**: the earlier data that can influence a particular solution value. To see its meaning over several steps, trace the stencil backward again. Two steps earlier, the FTBS value can depend on points $i$, $i-1$, and $i-2$; after $m$ steps, its numerical dependence extends at most from $x_{i-m}$ to $x_i$. The exact characteristic reaches $x_i-mc\Delta t$ over the same interval. We are considering points whose backward stencils remain inside the spatial domain; at an inflow boundary, the prescribed boundary data also enter the dependence.

For the characteristic to stay within the numerical dependence interval, we need

$$
\label{eq-cfl-geometric-condition}
mc\Delta t\leq m\Delta x,
\qquad\text{or}\qquad C\leq1.
$$

We have assumed positive speed and positive step sizes. The limiting case $C=0$ corresponds to no travel during a step. Notice that equality is allowed: traveling exactly one grid interval does not put the required information beyond the stencil.

If $C>1$, the exact wave travels farther during a fixed time interval than information can pass through the numerical stencil. Refining with that same supercritical Courant number does not remove the mismatch. The numerical method cannot, in general, converge to a solution that depends on data it cannot reach. This information-containment requirement is the **CFL condition** for the stencil considered here.

:::{warning .simple .dropdown icon=false open=false} On paper
Sketch the backward characteristic for $C=0.8$, $1.0$, and $1.2$, starting from the same new grid point. Mark the two old grid points used by FTBS. Where does each characteristic land? Then trace two numerical steps backward and identify the three old points that can influence the new value.
:::

The geometry explains why $C=1$ is a meaningful threshold, but it does not tell us how the update combines the available values. Containing the characteristic is a necessary condition for convergence of these explicit transport schemes; it is not, by itself, a proof that disturbances remain controlled. To explain the decay, preservation, and growth measured in our perturbation study, we will next examine the **weights** multiplying the two old values.